# Qwen Q8 Chat on Colab

A100 80GB 向けのチャット・画像理解・Web検索・音声入力専用Notebookです。画像生成・編集はありません。既定の Q8_K_L abliterated モデルと公式 mmproj を使用します。

1. A100 80GB ランタイムを選択します。
2. 必要に応じて Secrets に `HF_TOKEN` と検索 API キーを設定します。
3. Cells 1–4 を順に実行します。Cell 5 は利用終了時だけ使います。


In [ ]:
# Cell 1: リポジトリを取得
import os
import subprocess

REPO = "https://github.com/moruku36/qwen-multimodal-colab"
BRANCH = "feat/q8-chat-colab"  # draft PR branch; use main after merge
DEST = "/content/qwen-multimodal-colab"
if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST], check=True)
else:
    subprocess.run(["git", "-C", DEST, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", DEST, "pull", "--ff-only", "origin", BRANCH], check=True)
%cd /content/qwen-multimodal-colab
!git log --oneline -1

In [ ]:
# Cell 2: 依存関係（Colab標準のtorch/CUDAはそのまま）+ llama.cpp（2回目以降はDriveキャッシュから数秒）
%pip install -q -r requirements-chat-colab.txt
import sys

sys.path.insert(
    0, "/content/qwen-multimodal-colab/src"
)  # editable install is not visible in a running kernel
from qmc import colab

colab.setup()  # Drive マウント（llama.cpp のキャッシュ先にも使う）
if not os.environ.get("QMC_CHAT_BASE_URL"):
    colab.install_llama_cpp()

In [ ]:
# Cell 3: Q8 chat settings
SHARE = False  # public link if True; use authentication when sharing
PREFETCH = False  # chat GGUF and vision projector only
WEB_SEARCH = "on"  # on / auto / off
if PREFETCH:
    colab.prefetch_models(chat=True, image=False)
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
# Cell 4: launch; successful background UI stays connected
try:
    app = colab.launch(share=SHARE, profile="a100_80", web_search=WEB_SEARCH, chat_only=True)
except BaseException:
    try:
        colab.shutdown_runtime()
    except Exception as cleanup_error:
        print(f"Automatic cleanup failed; end the session manually: {cleanup_error}")
    raise


In [ ]:
# Cell 5: run only when finished; confirmation button releases GPU and ends runtime
from qmc import colab
colab.show_shutdown_button()


### Notes
- Q8 model: `huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF` / `Huihui-Qwen3.8-27B-abliterated-UD-DW-Q8_K_L.gguf`.
- Vision projector: `ggml-org/Qwen3.8-27B-GGUF` / `mmproj-Qwen3.8-27B-Q8_0.gguf`.
- The Release GPU model button unloads weights; it does not end Colab billing. Cell 5 explicitly ends the runtime.
- A100 performance, quality, memory, and Colab compute usage have not been measured for this variant. Validate on or after October 23.
